# 07 — Domain-stratified (proposal 3.3.7) → RQ4

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # so `import src...` works from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.utils import set_seed, save_fig
set_seed()  # fix all RNGs -- reproducibility

Re-run core classification per domain (8 of them). Report per-class F1, confusion, feature importance per domain.

In [ ]:
from src import data, features, analysis, viz
clean = data.load_or_build_clean(); splits = data.load_or_build_splits(clean)
texts = clean['text']

X_tfidf, _ = features.build_tfidf(texts.iloc[splits['train']], texts)
sty = features.build_stylometric(texts)
sty_scaled, _ = features.scale_dense(sty.values, splits['train'])
bib = features.build_biber(texts)
bib_scaled, _ = features.scale_dense(bib.values, splits['train'])
emb = features.build_sbert(texts)   # cached

feature_blocks = {'tfidf': X_tfidf, 'stylometric': sty_scaled, 'biber': bib_scaled, 'sbert': emb,
                   'length': clean[['log_token_count']].values}

In [ ]:
per_domain = analysis.run_per_domain(clean, splits, feature_blocks, 'logreg')
per_domain.to_csv(config.ARTIFACTS / 'per_domain_results.csv', index=False)

pivot = per_domain.pivot_table(index='domain', columns='class', values='f1').reindex(columns=config.CLASSES)
pivot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), facecolor=viz.SURFACE)
im = ax.imshow(pivot.values, cmap=viz.SEQUENTIAL_BLUE, vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index, fontsize=9)
ax.set_title('Per-domain, per-class F1 (Exp 6 features + logreg, retrained per domain)')
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('F1')
fig.tight_layout()
save_fig(fig, 'per_domain_f1_heatmap')

In [ ]:
domain_macro = per_domain.drop_duplicates('domain').set_index('domain')['macro_f1_domain'].reindex(config.DOMAINS)

fig, ax = viz.new_fig(figsize=(8, 5))
x = np.arange(len(domain_macro))
bars = ax.bar(x, domain_macro.values, color=viz.SEQUENTIAL_BLUE(0.6), zorder=3)
ax.bar_label(bars, fmt='%.2f', fontsize=8, color=viz.INK_SECONDARY, padding=2)
ax.set_xticks(x)
ax.set_xticklabels(domain_macro.index, rotation=20, ha='right')
ax.set_ylabel('Macro-F1')
ax.set_ylim(0, 1)
ax.set_title('Per-domain Macro-F1 (RQ4) — same feature set/classifier, retrained per domain')
fig.tight_layout()
save_fig(fig, 'per_domain_macro_f1')

Wrap up: collect the RQ1–RQ4 tables/figures for the report.

In [ ]:
print('Figures for the report:', sorted(p.name for p in config.FIGURES.glob('*.png')))
print('Result tables:', sorted(p.name for p in config.ARTIFACTS.glob('*.csv')))